In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql import SparkSession
from pyspark.sql import Row
import json
from pyspark.sql.functions import current_timestamp, trunc, add_months, date_format, col
import pytz
from datetime import datetime, timedelta
from pyspark.sql.window import Window
import pyspark.sql.functions as F
import pyspark.pandas as ps
from pyspark.sql.functions import col, explode, split
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType
from pyspark.sql.functions import col, lit, when, struct
from pyspark.sql.types import StructType, StructField, StringType, IntegerType , DateType , BooleanType , DoubleType ,TimestampType,ArrayType,ArrayType,LongType
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, DoubleType
from pyspark.sql.utils import *
spark = SparkSession.builder.appName("json-to-parquet").config("spark.driver.maxResultSize", "-1").getOrCreate()
from pyspark.sql.utils import AnalysisException
from pyspark.sql.functions import last_day
import pandas as pd
import numpy as np
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
from synapse.ml.core.platform import *
from delta.tables import DeltaTable

StatementMeta(, d3632eaf-1b89-40f5-a0a3-cab6b8ef095e, 3, Finished, Available, Finished, False)

/opt/spark/python/lib/pyspark.zip/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.


In [2]:
TotalPrem = 'abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DIEP2_IH.Lakehouse/Tables/TransactionTotalPremium_versions/'
Transaction_versions = 'abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DIEP2_IH.Lakehouse/Tables/Transaction_versions/'
Policy_versions = 'abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DIEP2_IH.Lakehouse/Tables/Policy_versions/'
Insured = 'abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DIEP2_IH.Lakehouse/Tables/InsuredAccount_versions/'
RiskAttributes = 'abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DIEP2_IH.Lakehouse/Tables/PolicyRiskAttributes_versions/'
BillingPolicyInterface = 'abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DEBIH.Lakehouse/Tables/UW1_POLICY_INTERFACE/'
Billingstatus = 'abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DEBIH.Lakehouse/Tables/UW2_STATUS_TYPE/'
Billingpolicy = 'abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DEBIH.Lakehouse/Tables/PR1_POLICY/'

TotalPrem = spark.read.format("delta").load(TotalPrem)
Transaction_versions=spark.read.format("delta").load(Transaction_versions)
Policy_versions=spark.read.format("delta").load(Policy_versions)
Insured=spark.read.format("delta").load(Insured)
RiskAttributes=spark.read.format("delta").load(RiskAttributes)
BillingPolicyInterface=spark.read.format("delta").load(BillingPolicyInterface)
Billingstatus=spark.read.format("delta").load(Billingstatus)
Billingpolicy=spark.read.format("delta").load(Billingpolicy)

TotalPrem.createOrReplaceTempView("TotalPrem")
Transaction_versions.createOrReplaceTempView("Transaction_versions")
Policy_versions.createOrReplaceTempView("Policy_versions")
Insured.createOrReplaceTempView("Insured")
RiskAttributes.createOrReplaceTempView("RiskAttributes")
BillingPolicyInterface.createOrReplaceTempView("BillingPolicyInterface")
Billingstatus.createOrReplaceTempView("Billingstatus")
Billingpolicy.createOrReplaceTempView("Billingpolicy")

StatementMeta(, d3632eaf-1b89-40f5-a0a3-cab6b8ef095e, 4, Finished, Available, Finished, False)

In [3]:
from datetime import datetime, timedelta
import pytz

CUTOFF_BUFFER_MINUTES = 30
utc_tz = pytz.utc

cutoff_time_utc = (
    datetime.now(utc_tz) - timedelta(minutes=CUTOFF_BUFFER_MINUTES)
).strftime('%Y-%m-%d %H:%M:%S')

print(f"Reconciliation cutoff (UTC): {cutoff_time_utc}")

StatementMeta(, d3632eaf-1b89-40f5-a0a3-cab6b8ef095e, 5, Finished, Available, Finished, False)

Reconciliation cutoff (UTC): 2026-06-29 13:37:54


In [4]:
# ============================================================
# EXCEPTION LIST — Policies excluded from reconciliation
# Only specific terms are excluded — renewal terms still included
# To add more: add a new row in exception_list below
# Format: (BasePolicyNumber, TermEffectiveDate, TermExpirationDate, Reason)
# ============================================================

exception_list = [
    # GAPA006654200-1 → exclude Term 2024-10-29 to 2026-04-29 only
    ("GAPA006654200-1", "2025-10-29", "2026-04-29", "Old transaction — client accepted difference"),
    # GAPA006750968 → exclude Term 2025-10-08 to 2026-04-08 only
    ("GAPA006750968",   "2025-10-08", "2026-04-08", "Old transaction — client accepted difference"),
    # Renewal term GAPA006750968-1 (2026-04-08 to 2026-10-08) is NOT excluded — still monitored
    # Add more exceptions here as needed
]

exception_df = spark.createDataFrame(
    [(p, e, x) for p, e, x, r in exception_list],
    ["PolicyNumber", "TermEffectiveDate", "TermExpirationDate"]
)

exception_df = exception_df.withColumn("TermEffectiveDate", F.to_date("TermEffectiveDate")) \
                           .withColumn("TermExpirationDate", F.to_date("TermExpirationDate"))

print("=== EXCEPTION LIST ===")
exception_df.show(truncate=False)
print(f"Total exceptions: {exception_df.count()}")

StatementMeta(, d3632eaf-1b89-40f5-a0a3-cab6b8ef095e, 6, Finished, Available, Finished, False)

=== EXCEPTION LIST ===
+---------------+-----------------+------------------+
|PolicyNumber   |TermEffectiveDate|TermExpirationDate|
+---------------+-----------------+------------------+
|GAPA006654200-1|2025-10-29       |2026-04-29        |
|GAPA006750968  |2025-10-08       |2026-04-08        |
+---------------+-----------------+------------------+

Total exceptions: 2


In [5]:
def get_policy_data():
    query = f"""
    SELECT *
FROM
(
    Select 
        pv.PolicyNumber,
        ra.BillingAccountNumber,
        CAST(pv.EffectiveDate as Date) AS EffectiveDate,
        tv.Number,
        tv.Date as TransactionDate,
        tv.Type as TransactionType,
        ROUND(tp.EffectivePremium,2) AS WrittenPremium,
        ROUND(tp.AnnualPremium,2) AS AnnualPremium,
        tv.OOSGroupNumber,
        ROW_NUMBER() OVER (
            PARTITION BY pv.PolicyNumber, tv.OOSGroupNumber
            ORDER BY tv.Number DESC,
                     CASE WHEN tv.status = 'Committed' THEN 0 ELSE 1 END ASC
        ) as rn,
        (NVL(tp.Fees,0) + NVL(tp.taxes,0)) AS feesntaxes,
        'Policy' as Source 

    FROM 
        Policy_versions pv
    JOIN 
        Transaction_versions tv ON pv.Policy_ref = tv.Policy_ref
    JOIN 
        TotalPrem tp ON pv.Policy_ref = tp.Policy_ref
    LEFT JOIN 
        Insured ia ON pv.Policy_ref = ia.Policy_ref
    LEFT JOIN
        RiskAttributes ra ON pv.Policy_ref = ra.Policy_ref    

    WHERE
        pv.PolicyStatus NOT IN (
            'Pending Cancellation',
            'Pending Cancellation-Signed Application',
            'Rescind',
            'Renewal-Offered',
            'Non-Renewal - Added',
            'Non-Renewal - Removed'
        )
        AND tv.Type != 'Non Renewals'
        AND tv.status = 'Committed'
        AND pv.`Audit.LastUpdatedOn` <= '{cutoff_time_utc}'
) t
ORDER BY Number, TransactionDate Desc;
    """
    return spark.sql(query)

StatementMeta(, d3632eaf-1b89-40f5-a0a3-cab6b8ef095e, 7, Finished, Available, Finished, False)

In [6]:
df_policy =get_policy_data()
#policy = spark.createDataFrame(get_policy_data().toPandas(),schema)
#policy = policy.withColumn('PolicyNumber',split(policy['PolicyNumber'],'-').getItem(0))
df_policy.show(25)

StatementMeta(, d3632eaf-1b89-40f5-a0a3-cab6b8ef095e, 8, Finished, Available, Finished, False)

+---------------+--------------------+-------------+------+---------------+---------------+--------------+-------------+--------------+---+----------+------+
|   PolicyNumber|BillingAccountNumber|EffectiveDate|Number|TransactionDate|TransactionType|WrittenPremium|AnnualPremium|OOSGroupNumber| rn|feesntaxes|Source|
+---------------+--------------------+-------------+------+---------------+---------------+--------------+-------------+--------------+---+----------+------+
|GAPA006709387-1|           100017524|   2026-06-28|     0|     2026-06-27|        Renewal|        1995.0|         1995|          NULL|  1|       0.0|Policy|
|GAPA006736668-1|           100017516|   2026-06-25|     0|     2026-06-27|        Renewal|        1280.0|         1280|          NULL|  1|       0.0|Policy|
|  GAPA006760876|           100017515|   2026-06-27|     0|     2026-06-27|         Policy|        1245.0|         1245|           0.0|  1|       0.0|Policy|
|  GAPA006760877|           100017517|   2026-06-27|

In [7]:
def get_billings_data():
    billingquery = f"""
    WITH policy_suffix_map AS (
        SELECT DISTINCT
            REGEXP_REPLACE(PolicyNumber, '-[0-9]+$', '') AS BaseNumber,
            PolicyNumber AS FullPolicyNumber,
            CAST(EffectiveDate AS DATE) AS EffectiveDate
        FROM policy_versions
    )
    Select 
        CASE 
            WHEN LEN(NVL(p.POL_NUMBER_IO,'')) > 0 THEN p.POL_NUMBER_IO
            WHEN pol_map.FullPolicyNumber IS NOT NULL THEN pol_map.FullPolicyNumber
            ELSE p.pol_number
        END AS PolicyNumber,
        pi.POLICY_INTERFACE_ACCOUNT_NUMBER_IN as BillingAccountNumber, 
        pi.POLICY_INTERFACE_EFFECTIVE_DATE as EffectiveDate,
        POLICY_INTERFACE_ENDORSEMENT as Number, 
        pi.POLICY_INTERFACE_ENDORSEMENT_DATE as TransactionDate,
        s.STATUS_TYPE_DESC as TransactionType,
        ROUND(pi.POLICY_INTERFACE_PREMIUM, 2) as WrittenPremium,
        pi.POLICY_INTERFACE_TOTAL_PREMIUM_ON_TERM as AnnualPremium,
        NVL(POLICY_INTERFACE_FEE1_VALUE,0) + NVL(POLICY_INTERFACE_FEE2_VALUE,0) +
        NVL(POLICY_INTERFACE_FEE3_VALUE,0) + NVL(POLICY_INTERFACE_FEE4_VALUE,0) +
        NVL(POLICY_INTERFACE_FEE5_VALUE,0) + NVL(POLICY_INTERFACE_FEE6_VALUE,0) +
        NVL(POLICY_INTERFACE_FEE7_VALUE,0) + NVL(POLICY_INTERFACE_FEE8_VALUE,0) +
        NVL(POLICY_INTERFACE_FEE9_VALUE,0) + NVL(POLICY_INTERFACE_FEE10_VALUE,0) +
        NVL(POLICY_INTERFACE_TAX1_VALUE,0) + NVL(POLICY_INTERFACE_TAX2_VALUE,0) +
        NVL(POLICY_INTERFACE_TAX3_VALUE,0) + NVL(POLICY_INTERFACE_TAX4_VALUE,0) +
        NVL(POLICY_INTERFACE_TAX5_VALUE,0) + NVL(POLICY_INTERFACE_TAX6_VALUE,0) AS feesntaxes,
        'Billing' as Source   
    FROM 
        BillingPolicyInterface pi
    LEFT JOIN    
        Billingpolicy p ON p.POL_NUMBER = pi.POLICY_INTERFACE_POLICY_NUM 
        AND p.pol_effective_date = pi.POLICY_INTERFACE_EFFECTIVE_DATE
    LEFT JOIN    
        Billingstatus s ON s.STATUS_TYPE_CODE = pi.POLICY_INTERFACE_STATUS_REASON
    LEFT JOIN
        policy_suffix_map pol_map
        ON pol_map.BaseNumber = pi.POLICY_INTERFACE_POLICY_NUM
        AND pol_map.EffectiveDate = pi.POLICY_INTERFACE_EFFECTIVE_DATE
    WHERE pi.POLICY_INTERFACE_TIME_STAMPC <= '{cutoff_time_utc}'
    ORDER BY pi.POLICY_INTERFACE_POLICY_NUM, pi.POLICY_INTERFACE_ENDORSEMENT    
    """
    return spark.sql(billingquery)

StatementMeta(, d3632eaf-1b89-40f5-a0a3-cab6b8ef095e, 9, Finished, Available, Finished, False)

In [8]:
df_billing=get_billings_data()
#billing =spark.createDataFrame(get_billings_data().toPandas(),schema) 
df_billing.show(25)


StatementMeta(, d3632eaf-1b89-40f5-a0a3-cab6b8ef095e, 10, Finished, Available, Finished, False)

+---------------+--------------------+-------------+------+---------------+------------------+--------------+-------------+----------+-------+
|   PolicyNumber|BillingAccountNumber|EffectiveDate|Number|TransactionDate|   TransactionType|WrittenPremium|AnnualPremium|feesntaxes| Source|
+---------------+--------------------+-------------+------+---------------+------------------+--------------+-------------+----------+-------+
|  GAPA000000002|           100000001|   2025-06-12|     0|     2025-06-12|               New|        851.00|       851.00|   30.0000|Billing|
|  GAPA000000002|           100000001|   2025-06-12|     1|     2025-06-13|       Endorsement|          0.00|       851.00|    0.0000|Billing|
|GAPA000000002-1|           100000001|   2025-12-12|     2|     2025-12-12|           Renewal|        760.00|       760.00|   30.0000|Billing|
|GAPA000000002-2|           100000001|   2026-06-12|     3|     2026-06-12|           Renewal|        758.00|       758.00|   30.0000|Billing|

In [9]:
df_policy = df_policy.withColumn(
    "WrittenPremium", F.round("WrittenPremium", 2)
)

df_billing = df_billing.withColumn(
    "WrittenPremium", F.round("WrittenPremium", 2)
)

StatementMeta(, d3632eaf-1b89-40f5-a0a3-cab6b8ef095e, 11, Finished, Available, Finished, False)

In [10]:
df_policy.select(
    "PolicyNumber", "BillingAccountNumber", "WrittenPremium", "Number"
).filter(
    F.col("PolicyNumber") == "GAPA006757497"
).show(truncate=False)

StatementMeta(, d3632eaf-1b89-40f5-a0a3-cab6b8ef095e, 12, Finished, Available, Finished, False)

+-------------+--------------------+--------------+------+
|PolicyNumber |BillingAccountNumber|WrittenPremium|Number|
+-------------+--------------------+--------------+------+
|GAPA006757497|100013466           |1100.0        |0     |
|GAPA006757497|100013466           |0.0           |1     |
|GAPA006757497|100013466           |-788.69       |2     |
|GAPA006757497|100013466           |788.69        |3     |
+-------------+--------------------+--------------+------+



In [11]:
df_billing.select(
    "PolicyNumber", "BillingAccountNumber", "WrittenPremium", "Number"
).filter(
    F.col("PolicyNumber") == "GAPA006757497"
).show(truncate=False)

StatementMeta(, d3632eaf-1b89-40f5-a0a3-cab6b8ef095e, 13, Finished, Available, Finished, False)

+-------------+--------------------+--------------+------+
|PolicyNumber |BillingAccountNumber|WrittenPremium|Number|
+-------------+--------------------+--------------+------+
|GAPA006757497|100013466           |1100.00       |0     |
|GAPA006757497|100013466           |0.00          |1     |
|GAPA006757497|100013466           |-788.69       |2     |
|GAPA006757497|100013466           |788.69        |3     |
+-------------+--------------------+--------------+------+



In [12]:
total_billing_rows = df_billing.count()
total_policy_rows = df_policy.count()

StatementMeta(, d3632eaf-1b89-40f5-a0a3-cab6b8ef095e, 14, Finished, Available, Finished, False)

In [13]:
# =========================
# 1. AGGREGATE STATS
# =========================

policy_stats = df_policy.groupBy("PolicyNumber") \
    .agg(
        F.count("*").alias("policy_cnt"),
        F.round(F.sum("WrittenPremium"), 2).alias("policy_sum"),
        F.max("WrittenPremium").alias("policy_max"),
        F.min("WrittenPremium").alias("policy_min")
    )

billing_stats = df_billing.groupBy("PolicyNumber") \
    .agg(
        F.count("*").alias("billing_cnt"),
        F.round(F.sum("WrittenPremium"), 2).alias("billing_sum"),
        F.max("WrittenPremium").alias("billing_max"),
        F.min("WrittenPremium").alias("billing_min")
    )

stats = policy_stats.join(
    billing_stats,
    ["PolicyNumber"],
    "outer"
).fillna(0)


StatementMeta(, d3632eaf-1b89-40f5-a0a3-cab6b8ef095e, 15, Finished, Available, Finished, False)

In [14]:
# =========================
# 2. FLAGS + VARIANCE
# =========================

stats_flagged = (
    stats
    # ---- Variance ----
    .withColumn("PolicyTotalPremium", F.col("policy_sum"))
    .withColumn("BillingTotalPremium", F.col("billing_sum"))
    .withColumn("Variance", F.round(F.col("policy_sum") - F.col("billing_sum"), 2))
    .withColumn("AbsVariance", F.abs(F.col("Variance")))

    # ---- Variance bucket ----
    .withColumn(
        "VarianceBucket",
        F.when(F.abs(F.col("Variance")) == 0, "No Variance")
         .when(F.abs(F.col("Variance")) < 10, "Low")
         .when(F.abs(F.col("Variance")) < 100, "Medium")
         .otherwise("High")
    )
)


StatementMeta(, d3632eaf-1b89-40f5-a0a3-cab6b8ef095e, 16, Finished, Available, Finished, False)

In [15]:
# ============================================================
# APPLY EXCEPTION — Remove excepted policy terms from dashboard
# ============================================================
stats_flagged = stats_flagged.join(
    exception_df.select("PolicyNumber"),
    "PolicyNumber",
    "left_anti"
)
print(f"Mismatches after exception filter: {stats_flagged.filter(F.col('Variance') != 0).count()}")

StatementMeta(, d3632eaf-1b89-40f5-a0a3-cab6b8ef095e, 17, Finished, Available, Finished, False)

Mismatches after exception filter: 453


In [16]:
stats_flagged.select("PolicyNumber","BillingTotalPremium","PolicyTotalPremium","Variance").filter(
    F.col("PolicyNumber") == "GAPA006757497"
).show(truncate=False)

StatementMeta(, d3632eaf-1b89-40f5-a0a3-cab6b8ef095e, 18, Finished, Available, Finished, False)

+-------------+-------------------+------------------+--------+
|PolicyNumber |BillingTotalPremium|PolicyTotalPremium|Variance|
+-------------+-------------------+------------------+--------+
|GAPA006757497|1100.00            |1100.0            |0.0     |
+-------------+-------------------+------------------+--------+



In [17]:
# 1. Define the value columns (ignoring Date and Number)
value_cols = ["PolicyNumber", "WrittenPremium"]

# 2. Assign an "Occurrence Number" to handle identical transactions
# This tells us: "This is the 1st time we've seen $10.54$, this is the 2nd time..."
w = Window.partitionBy(*value_cols).orderBy(F.col("TransactionDate").asc_nulls_last(),
    F.col("EffectiveDate").asc_nulls_last()
)

df_b_counted = df_billing.withColumn("occurrence", F.row_number().over(w))
df_p_counted = df_policy.withColumn("occurrence", F.row_number().over(w))

# 3. Perform the Anti-Join using Value + Occurrence
# Now, the 1st $10.54$ in Billing matches the 1st $10.54$ in Policy and is removed.
# The 2nd $10.54$ in Billing (Row 10) has no partner in Policy!
only_in_billing = df_b_counted.join(
    df_p_counted, 
    on=value_cols + ["occurrence"], 
    how="left_anti"
)

desired_cols = [
    "PolicyNumber", "BillingAccountNumber", "EffectiveDate",
    "TransactionDate", "Number", "WrittenPremium",
    "AnnualPremium", "feesntaxes", "occurrence", "Source"
]

# Keep only columns that actually exist in the DataFrame
existing_cols = [col for col in desired_cols if col in only_in_billing.columns]

# Select only existing columns
only_in_billing = only_in_billing.select(*existing_cols)

only_in_billing.show()

StatementMeta(, d3632eaf-1b89-40f5-a0a3-cab6b8ef095e, 19, Finished, Available, Finished, False)

+---------------+--------------------+-------------+---------------+------+--------------+-------------+----------+----------+-------+
|   PolicyNumber|BillingAccountNumber|EffectiveDate|TransactionDate|Number|WrittenPremium|AnnualPremium|feesntaxes|occurrence| Source|
+---------------+--------------------+-------------+---------------+------+--------------+-------------+----------+----------+-------+
|GAPA005003635-1|           100003537|   2025-10-24|     2026-03-16|     4|        -45.36|       713.98|    0.0000|         1|Billing|
|GAPA005004984-1|           100001569|   2025-09-20|     2025-12-04|     3|       -845.00|      1165.26|    0.0000|         1|Billing|
|GAPA005007724-1|           100009964|   2026-01-21|     2026-07-21|     4|          0.00|      1363.60|    0.0000|         1|Billing|
|GAPA005008700-1|           100001465|   2025-09-16|     2026-03-10|     4|         -9.50|      1944.18|    0.0000|         2|Billing|
|GAPA005009699-1|           100008032|   2025-12-28|   

In [18]:
only_in_policy = df_p_counted.join(
    df_b_counted, 
    on=value_cols + ["occurrence"], 
    how="left_anti"
)

desired_cols = [
    "PolicyNumber", "BillingAccountNumber", "EffectiveDate",
    "TransactionDate", "Number", "WrittenPremium",
    "AnnualPremium", "feesntaxes", "occurrence", "Source"
]

# Keep only columns that actually exist in the DataFrame
existing_cols = [col for col in desired_cols if col in only_in_policy.columns]

# Select only existing columns
only_in_policy = only_in_policy.select(*existing_cols)

only_in_policy.show()

StatementMeta(, d3632eaf-1b89-40f5-a0a3-cab6b8ef095e, 20, Finished, Available, Finished, False)

+---------------+--------------------+-------------+---------------+------+--------------+-------------+----------+----------+------+
|   PolicyNumber|BillingAccountNumber|EffectiveDate|TransactionDate|Number|WrittenPremium|AnnualPremium|feesntaxes|occurrence|Source|
+---------------+--------------------+-------------+---------------+------+--------------+-------------+----------+----------+------+
|GAPA004011546-2|           100004358|   2026-05-15|     2026-06-26|     1|         550.0|          550|       0.0|         1|Policy|
|GAPA005003635-1|           100003537|   2025-10-24|     2026-03-16|     3|           0.0|          714|       0.0|         2|Policy|
|GAPA005006095-2|           100007653|   2026-06-27|     2026-06-25|     1|         776.0|          776|       0.0|         1|Policy|
|GAPA005006453-2|           100001065|   2026-03-12|     2026-06-26|     3|        244.65|         1565|       0.0|         1|Policy|
|GAPA005009320-2|           100007966|   2026-06-27|     2026-

In [19]:
valid_keys = stats_flagged.filter(
    (F.col("Variance").cast(IntegerType())) !=0
).select("PolicyNumber")

StatementMeta(, d3632eaf-1b89-40f5-a0a3-cab6b8ef095e, 21, Finished, Available, Finished, False)

In [20]:
# =========================
# 3. JOIN FLAGS TO MISMATCH DATA
# =========================

common_cols = [
    "PolicyNumber",
    "PolicyTotalPremium",
    "BillingTotalPremium",
    "Variance",
    "AbsVariance",
    "VarianceBucket",
    "policy_cnt",
    "billing_cnt"
]

only_in_policy_flagged = only_in_policy.join(
    stats_flagged.select(*common_cols),
    ["PolicyNumber"],
    "left"
)

only_in_billing_flagged = only_in_billing.join(
    stats_flagged.select(*common_cols),
    ["PolicyNumber"],
    "left"
)

StatementMeta(, d3632eaf-1b89-40f5-a0a3-cab6b8ef095e, 23, Finished, Available, Finished, False)

In [21]:
# policy_mismatch_net = only_in_policy.groupBy("PolicyNumber", "BillingAccountNumber") \
#     .agg(
#         F.round(F.sum("WrittenPremium"), 2).alias("policy_sum"),
#         F.max("WrittenPremium").alias("policy_max"),
#         F.min("WrittenPremium").alias("policy_min")
#     )

# billing_mismatch_net = only_in_billing.groupBy("PolicyNumber", "BillingAccountNumber") \
#     .agg(
#         F.round(F.sum("WrittenPremium"), 2).alias("billing_sum"),
#         F.max("WrittenPremium").alias("billing_max"),
#         F.min("WrittenPremium").alias("billing_min")
#     )

StatementMeta(, d3632eaf-1b89-40f5-a0a3-cab6b8ef095e, 24, Finished, Available, Finished, False)

In [22]:
# net_mismatch = policy_mismatch_net.join(
#     billing_mismatch_net,
#     ["PolicyNumber", "BillingAccountNumber"],
#     "outer"
# ).fillna(0)

StatementMeta(, d3632eaf-1b89-40f5-a0a3-cab6b8ef095e, 25, Finished, Available, Finished, False)

In [23]:
# net_flag = net_mismatch.withColumn(
#     "is_reversal_case",
#     F.when(
#         (F.abs(F.col("policy_sum")) < 0.01) &
#         (F.abs(F.col("billing_sum")) < 0.01) &
#         (
#             # one side has reversal
#             (
#                 (F.col("billing_max") > 0) & (F.col("billing_min") < 0)
#             )
#             |
#             (
#                 (F.col("policy_max") > 0) & (F.col("policy_min") < 0)
#             )
#         ),
#         1
#     ).otherwise(0)
# )

StatementMeta(, d3632eaf-1b89-40f5-a0a3-cab6b8ef095e, 26, Finished, Available, Finished, False)

In [24]:
# reversal_pairs = net_flag.filter(
#     F.col("is_reversal_case") == 1
# ).select("PolicyNumber", "BillingAccountNumber").distinct()

StatementMeta(, d3632eaf-1b89-40f5-a0a3-cab6b8ef095e, 27, Finished, Available, Finished, False)

In [25]:
# only_in_policy_clean = only_in_policy.join(
#     reversal_pairs,
#     ["PolicyNumber", "BillingAccountNumber"],
#     "left_anti"
# )

# only_in_billing_clean = only_in_billing.join(
#     reversal_pairs,
#     ["PolicyNumber", "BillingAccountNumber"],
#     "left_anti"
# )

StatementMeta(, d3632eaf-1b89-40f5-a0a3-cab6b8ef095e, 28, Finished, Available, Finished, False)

In [26]:
dup_window = Window.partitionBy(
    "PolicyNumber",
    "WrittenPremium",
    "EffectiveDate",
    "TransactionDate",
    "Source"
).orderBy("occurrence")  # or any stable column

only_in_policy_flagged = only_in_policy_flagged.withColumn(
    "dup_occurrence",
    F.row_number().over(dup_window)
)

only_in_billing_flagged = only_in_billing_flagged.withColumn(
    "dup_occurrence",
    F.row_number().over(dup_window)
)

StatementMeta(, d3632eaf-1b89-40f5-a0a3-cab6b8ef095e, 29, Finished, Available, Finished, False)

In [85]:
only_in_policy_flagged = only_in_policy_flagged.withColumn(
    "IsDuplicate",
    F.when(F.col("dup_occurrence") > 1, 1).otherwise(0)
)

only_in_billing_flagged = only_in_billing_flagged.withColumn(
    "IsDuplicate",
    F.when(F.col("dup_occurrence") > 1, 1).otherwise(0)
)

StatementMeta(, f02b8f7e-465c-4e55-bffb-6218b04fc486, 92, Finished, Available, Finished, False)

In [86]:
only_in_policy_flagged.show()
only_in_billing_flagged.show()

StatementMeta(, f02b8f7e-465c-4e55-bffb-6218b04fc486, 93, Finished, Available, Finished, False)

+---------------+--------------------+-------------+---------------+------+--------------+-------------+----------+----------+------+------------------+-------------------+--------+-----------+--------------+----------+-----------+--------------+-----------+
|   PolicyNumber|BillingAccountNumber|EffectiveDate|TransactionDate|Number|WrittenPremium|AnnualPremium|feesntaxes|occurrence|Source|PolicyTotalPremium|BillingTotalPremium|Variance|AbsVariance|VarianceBucket|policy_cnt|billing_cnt|dup_occurrence|IsDuplicate|
+---------------+--------------------+-------------+---------------+------+--------------+-------------+----------+----------+------+------------------+-------------------+--------+-----------+--------------+----------+-----------+--------------+-----------+
|GAPA004011546-2|           100004358|   2026-05-15|     2026-06-26|     1|         550.0|          550|       0.0|         1|Policy|             550.0|               0.00|   550.0|      550.0|          High|         1|    

In [87]:
mismatch_df = only_in_policy_flagged.unionByName(only_in_billing_flagged)

mismatch_df = mismatch_df.join(
    valid_keys,
    ["PolicyNumber"],
    "inner"
)

df_flag = mismatch_df.withColumn(
    "abs_amt", F.abs(F.col("WrittenPremium"))
)

StatementMeta(, f02b8f7e-465c-4e55-bffb-6218b04fc486, 94, Finished, Available, Finished, False)

In [88]:
reversal_pairs = df_flag.groupBy(
    "PolicyNumber", "abs_amt"
).agg(
    F.max(F.when(F.col("WrittenPremium") > 0, 1).otherwise(0)).alias("has_pos"),
    F.max(F.when(F.col("WrittenPremium") < 0, 1).otherwise(0)).alias("has_neg")
).filter(
    (F.col("has_pos") == 1) & (F.col("has_neg") == 1)
).select(
    "PolicyNumber", "abs_amt"
)

StatementMeta(, f02b8f7e-465c-4e55-bffb-6218b04fc486, 95, Finished, Available, Finished, False)

In [89]:
reversal_policy = reversal_pairs.select(
    "PolicyNumber"
).distinct()

StatementMeta(, f02b8f7e-465c-4e55-bffb-6218b04fc486, 96, Finished, Available, Finished, False)

In [90]:
#Create flag lookup table
flags_df = df_flag.select(
    "PolicyNumber", "WrittenPremium"
).withColumn(
    "abs_amt", F.abs(F.col("WrittenPremium"))
).join(
    reversal_pairs.withColumn("IsReversal", F.lit(1)),
    ["PolicyNumber", "abs_amt"],
    "left"
).join(
    reversal_policy.withColumn("HasReversalPair", F.lit(1)),
    ["PolicyNumber"],
    "left"
).fillna({
    "IsReversal": 0,
    "HasReversalPair": 0
})

# =========================
# 4. ZERO REVERSAL FLAG (ROW LEVEL)
# =========================
flags_df = flags_df.withColumn(
    "IsZeroReversal",
    F.when(
        (F.col("WrittenPremium") == 0) &
        (F.col("HasReversalPair") == 1) &
        (F.col("IsReversal") == 0),
        1
    ).otherwise(0)
)

only_in_billing_flagged = only_in_billing_flagged.join(
    flags_df.select(
        "PolicyNumber",
        "WrittenPremium",
        "IsReversal",
        "IsZeroReversal"
    ),
    ["PolicyNumber", "WrittenPremium"],
    "left"
).fillna({
    "IsReversal": 0,
    "IsZeroReversal": 0
})

flags_clean = flags_df.groupBy(
    "PolicyNumber", "WrittenPremium"
).agg(
    F.max("IsReversal").alias("flag_IsReversal"),
    F.max("IsZeroReversal").alias("flag_IsZeroReversal")
)

# =========================
# 5. Rejoin to flagged tables
# =========================
only_in_billing_flagged = only_in_billing_flagged.join(
    flags_clean,
    ["PolicyNumber", "WrittenPremium"],
    "left"
)
only_in_billing_flagged = only_in_billing_flagged \
    .withColumn("IsReversal", F.col("IsReversal")) \
    .withColumn("IsZeroReversal", F.col("IsZeroReversal")) \
    .drop("flag_IsReversal", "flag_IsZeroReversal")

only_in_policy_flagged = only_in_policy_flagged.join(
    flags_clean,
    ["PolicyNumber", "WrittenPremium"],
    "left"
)

only_in_policy_flagged= only_in_policy_flagged \
    .withColumn("IsReversal", F.col("flag_IsReversal")) \
    .withColumn("IsZeroReversal", F.col("flag_IsZeroReversal")) \
    .drop("flag_IsReversal", "flag_IsZeroReversal")

# =========================
# 6. FINAL CLASSIFICATION
# =========================

def add_category(df):
    return df.withColumn(
        "ReconciliationCategory",
        F.when(F.col("IsReversal") == 1, "Reversal")
         .when(F.col("IsZeroReversal") == 1, "Zero Reversal")
         .when(F.col("IsDuplicate") == 1, "Duplicate")
         .otherwise("True Mismatch")
    )

only_in_policy_flagged = add_category(only_in_policy_flagged)
only_in_billing_flagged = add_category(only_in_billing_flagged)

StatementMeta(, f02b8f7e-465c-4e55-bffb-6218b04fc486, 97, Finished, Available, Finished, False)

In [91]:
only_in_policy_flagged= only_in_policy_flagged.withColumn("TotalPolicyRows", lit(total_policy_rows))
only_in_billing_flagged = only_in_billing_flagged.withColumn("TotalBillingRows", lit(total_billing_rows)) 

StatementMeta(, f02b8f7e-465c-4e55-bffb-6218b04fc486, 98, Finished, Available, Finished, False)

In [92]:
only_in_policy_flagged.count()

StatementMeta(, f02b8f7e-465c-4e55-bffb-6218b04fc486, 99, Finished, Available, Finished, False)

850

In [93]:
only_in_policy_flagged= only_in_policy_flagged.filter(F.col("Variance").cast(IntegerType()) != 0)
only_in_billing_flagged = only_in_billing_flagged.filter(F.col("Variance").cast(IntegerType()) != 0)

StatementMeta(, f02b8f7e-465c-4e55-bffb-6218b04fc486, 100, Finished, Available, Finished, False)

In [94]:
only_in_policy_flagged.count()

StatementMeta(, f02b8f7e-465c-4e55-bffb-6218b04fc486, 101, Finished, Available, Finished, False)

472

In [95]:
# only_in_policy=only_in_policy.groupby("PolicyNumber","Number")
only_in_policy_flagged = only_in_policy_flagged.orderBy(["PolicyNumber","Number"],ascending=[True, True])
only_in_policy_flagged.show()

StatementMeta(, f02b8f7e-465c-4e55-bffb-6218b04fc486, 102, Finished, Available, Finished, False)

+---------------+--------------+--------------------+-------------+---------------+------+-------------+----------+----------+------+------------------+-------------------+--------+-----------+--------------+----------+-----------+--------------+-----------+----------+--------------+----------------------+---------------+
|   PolicyNumber|WrittenPremium|BillingAccountNumber|EffectiveDate|TransactionDate|Number|AnnualPremium|feesntaxes|occurrence|Source|PolicyTotalPremium|BillingTotalPremium|Variance|AbsVariance|VarianceBucket|policy_cnt|billing_cnt|dup_occurrence|IsDuplicate|IsReversal|IsZeroReversal|ReconciliationCategory|TotalPolicyRows|
+---------------+--------------+--------------------+-------------+---------------+------+-------------+----------+----------+------+------------------+-------------------+--------+-----------+--------------+----------+-----------+--------------+-----------+----------+--------------+----------------------+---------------+
|GAPA004011546-2|         55

In [96]:
only_in_billing_flagged.count()

StatementMeta(, f02b8f7e-465c-4e55-bffb-6218b04fc486, 103, Finished, Available, Finished, False)

32

In [97]:
# only_in_billing=only_in_billing.groupby("PolicyNumber","Number")
only_in_billing_flagged = only_in_billing_flagged .orderBy(["PolicyNumber","Number"],ascending=[True, True])
only_in_billing_flagged.show()

StatementMeta(, f02b8f7e-465c-4e55-bffb-6218b04fc486, 104, Finished, Available, Finished, False)

+---------------+--------------+--------------------+-------------+---------------+------+-------------+----------+----------+-------+------------------+-------------------+--------+-----------+--------------+----------+-----------+--------------+-----------+----------+--------------+----------------------+----------------+
|   PolicyNumber|WrittenPremium|BillingAccountNumber|EffectiveDate|TransactionDate|Number|AnnualPremium|feesntaxes|occurrence| Source|PolicyTotalPremium|BillingTotalPremium|Variance|AbsVariance|VarianceBucket|policy_cnt|billing_cnt|dup_occurrence|IsDuplicate|IsReversal|IsZeroReversal|ReconciliationCategory|TotalBillingRows|
+---------------+--------------+--------------------+-------------+---------------+------+-------------+----------+----------+-------+------------------+-------------------+--------+-----------+--------------+----------+-----------+--------------+-----------+----------+--------------+----------------------+----------------+
|GAPA006566879-1|     

In [98]:
only_in_policy_flagged.select(
    "PolicyNumber", "WrittenPremium", "Number"
).filter(
    F.col("PolicyNumber") == "GAPA006716979-1"
).show(truncate=False)

StatementMeta(, f02b8f7e-465c-4e55-bffb-6218b04fc486, 105, Finished, Available, Finished, False)

+------------+--------------+------+
|PolicyNumber|WrittenPremium|Number|
+------------+--------------+------+
+------------+--------------+------+



In [99]:
only_in_billing_flagged.select(
    "PolicyNumber", "WrittenPremium", "Number"
).filter(
    F.col("PolicyNumber") == "GAPA006716979-1"
).show(truncate=False)

StatementMeta(, f02b8f7e-465c-4e55-bffb-6218b04fc486, 106, Finished, Available, Finished, False)

+------------+--------------+------+
|PolicyNumber|WrittenPremium|Number|
+------------+--------------+------+
+------------+--------------+------+



In [100]:
policyschema  = StructType([

# ---- Base transaction fields ----
StructField("PolicyNumber", StringType(), True),
StructField("EffectiveDate", DateType(), True),
StructField("TransactionDate", DateType(), True),
StructField("WrittenPremium", DoubleType(), True),
StructField("AnnualPremium", DoubleType(), True),
StructField("feesntaxes", DoubleType(), True),
StructField("occurrence", IntegerType(), True),
StructField("Source", StringType(), True),

# ---- Aggregates ----
StructField("PolicyTotalPremium", DoubleType(), True),
StructField("BillingTotalPremium", DoubleType(), True),
StructField("Variance", DoubleType(), True),
StructField("AbsVariance", DoubleType(), True),
StructField("VarianceBucket", StringType(), True),

# ---- Counts ----
StructField("policy_cnt", IntegerType(), True),
StructField("TotalPolicyRows", IntegerType(), True),

# ---- Flags ----
StructField("IsReversal", IntegerType(), True),
StructField("IsDuplicate", IntegerType(), True),
StructField("IsZeroReversal", IntegerType(), True),

# ---- Final classification ----
StructField("ReconciliationCategory", StringType(), True)

])

StatementMeta(, f02b8f7e-465c-4e55-bffb-6218b04fc486, 107, Finished, Available, Finished, False)

In [101]:
Billingschema  = StructType([

# ---- Base transaction fields ----
StructField("PolicyNumber", StringType(), True),
StructField("EffectiveDate", DateType(), True),
StructField("TransactionDate", DateType(), True),
StructField("WrittenPremium", DoubleType(), True),
StructField("AnnualPremium", DoubleType(), True),
StructField("feesntaxes", DoubleType(), True),
StructField("occurrence", IntegerType(), True),
StructField("Source", StringType(), True),

# ---- Aggregates ----
StructField("PolicyTotalPremium", DoubleType(), True),
StructField("BillingTotalPremium", DoubleType(), True),
StructField("Variance", DoubleType(), True),
StructField("AbsVariance", DoubleType(), True),
StructField("VarianceBucket", StringType(), True),

# ---- Counts ----
StructField("billing_cnt", IntegerType(), True),
StructField("TotalBillingRows", IntegerType(), True),
# ---- Flags ----
StructField("IsReversal", IntegerType(), True),
StructField("IsDuplicate", IntegerType(), True),
StructField("IsZeroReversal", IntegerType(), True),


# ---- Final classification ----
StructField("ReconciliationCategory", StringType(), True)

])

StatementMeta(, f02b8f7e-465c-4e55-bffb-6218b04fc486, 108, Finished, Available, Finished, False)

In [102]:
if only_in_policy is None:
    # If the join failed entirely, create a truly empty DF with your schema
    df_OnlyPolicy = spark.createDataFrame([], schema)
else:
    # 2. Align the existing DataFrame to your schema
    # This casts types and orders columns correctly without "iterating"
    df_OnlyPolicy = only_in_policy_flagged.select([
        F.col(field.name).cast(field.dataType) for field in policyschema.fields
    ])


StatementMeta(, f02b8f7e-465c-4e55-bffb-6218b04fc486, 109, Finished, Available, Finished, False)

In [103]:
if only_in_billing is None:
    # If the join failed entirely, create a truly empty DF with your schema
    df_OnlyBilling = spark.createDataFrame([], schema)
else:
    # 2. Align the existing DataFrame to your schema
    # This casts types and orders columns correctly without "iterating"
    df_OnlyBilling = only_in_billing_flagged.select([
        F.col(field.name).cast(field.dataType) for field in Billingschema.fields
    ])

StatementMeta(, f02b8f7e-465c-4e55-bffb-6218b04fc486, 110, Finished, Available, Finished, False)

In [104]:
df_OnlyPolicy.write.mode('overwrite').format('delta').option("overwriteSchema", "true").save('abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DIEP2_IH.Lakehouse/Tables/OnlyPolicy')
df_OnlyBilling.write.mode('overwrite').format('delta').option("overwriteSchema", "true").save('abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DIEP2_IH.Lakehouse/Tables/OnlyBilling')

StatementMeta(, f02b8f7e-465c-4e55-bffb-6218b04fc486, 111, Finished, Available, Finished, False)

In [105]:
events = [1, 2, 2, 3, 4, 4, 4]
from collections import Counter
counts = Counter(events)
print(dict(counts))

StatementMeta(, f02b8f7e-465c-4e55-bffb-6218b04fc486, 112, Finished, Available, Finished, False)

{1: 1, 2: 2, 3: 1, 4: 3}


In [106]:
counts = {}
for user_id in events:
    counts[user_id] = counts.get(user_id, 0) + 1

print(counts)

StatementMeta(, f02b8f7e-465c-4e55-bffb-6218b04fc486, 113, Finished, Available, Finished, False)

{1: 1, 2: 2, 3: 1, 4: 3}


In [107]:
print("=== SOURCE TABLE ROW COUNTS ===")
print(f"Policy_versions:       {spark.read.format('delta').load('abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DIEP2_IH.Lakehouse/Tables/Policy_versions/').count()}")
print(f"Transaction_versions:  {spark.read.format('delta').load('abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DIEP2_IH.Lakehouse/Tables/Transaction_versions/').count()}")
print(f"BillingPolicyInterface:{spark.read.format('delta').load('abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DEBIH.Lakehouse/Tables/UW1_POLICY_INTERFACE/').count()}")

print()
print("=== AFTER QUERY ROW COUNTS ===")
print(f"df_policy rows:  {df_policy.count()}")
print(f"df_billing rows: {df_billing.count()}")

print()
print("=== CUTOFF VALUE ===")
try:
    print(f"cutoff_time_utc: {cutoff_time_utc}")
except:
    print("cutoff_time_utc: NOT DEFINED")

print()
print("=== MISMATCH COUNT ===")
distinct_mismatched = stats_flagged.filter(F.col("Variance") != 0) \
                                    .select("PolicyNumber").distinct().count()
print(f"Distinct mismatched policies: {distinct_mismatched}")

print()
print("=== VARIANCE BUCKET BREAKDOWN ===")
stats_flagged.groupBy("VarianceBucket").count().show()

StatementMeta(, f02b8f7e-465c-4e55-bffb-6218b04fc486, 114, Finished, Available, Finished, False)

=== SOURCE TABLE ROW COUNTS ===
Policy_versions:       107330
Transaction_versions:  107330
BillingPolicyInterface:46612

=== AFTER QUERY ROW COUNTS ===
df_policy rows:  46532
df_billing rows: 46612

=== CUTOFF VALUE ===
cutoff_time_utc: 2026-06-29 09:32:45

=== MISMATCH COUNT ===
Distinct mismatched policies: 444

=== VARIANCE BUCKET BREAKDOWN ===
+--------------+-----+
|VarianceBucket|count|
+--------------+-----+
|          High|  380|
|           Low|   13|
|        Medium|   51|
|   No Variance|21118|
+--------------+-----+



In [108]:
print("=== SOURCE TABLE ROW COUNTS ===")
print(f"Policy_versions:       {spark.read.format('delta').load('abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DIEP2_IH.Lakehouse/Tables/Policy_versions/').count()}")
print(f"Transaction_versions:  {spark.read.format('delta').load('abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DIEP2_IH.Lakehouse/Tables/Transaction_versions/').count()}")
print(f"BillingPolicyInterface:{spark.read.format('delta').load('abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DEBIH.Lakehouse/Tables/UW1_POLICY_INTERFACE/').count()}")

print()
print("=== AFTER QUERY ROW COUNTS ===")
print(f"df_policy rows:  {df_policy.count()}")
print(f"df_billing rows: {df_billing.count()}")

print()
print("=== CUTOFF VALUE ===")
try:
    print(f"cutoff_time_utc: {cutoff_time_utc}")
except:
    print("cutoff_time_utc: NOT DEFINED")

print()
print("=== MISMATCH COUNT ===")
distinct_mismatched = stats_flagged.filter(F.col("Variance") != 0) \
                                    .select("PolicyNumber").distinct().count()
print(f"Distinct mismatched policies: {distinct_mismatched}")

print()
print("=== VARIANCE BUCKET BREAKDOWN ===")
stats_flagged.groupBy("VarianceBucket").count().show()

StatementMeta(, f02b8f7e-465c-4e55-bffb-6218b04fc486, 115, Finished, Available, Finished, False)

=== SOURCE TABLE ROW COUNTS ===
Policy_versions:       107330
Transaction_versions:  107330
BillingPolicyInterface:46612

=== AFTER QUERY ROW COUNTS ===
df_policy rows:  46532
df_billing rows: 46612

=== CUTOFF VALUE ===
cutoff_time_utc: 2026-06-29 09:32:45

=== MISMATCH COUNT ===
Distinct mismatched policies: 444

=== VARIANCE BUCKET BREAKDOWN ===
+--------------+-----+
|VarianceBucket|count|
+--------------+-----+
|          High|  380|
|           Low|   13|
|        Medium|   51|
|   No Variance|21118|
+--------------+-----+



In [109]:
print("=== LATEST BILLING TIMESTAMP ===")
spark.sql("""
    SELECT 
        MAX(POLICY_INTERFACE_TIME_STAMPC) AS latest_billing_timestamp,
        COUNT(*) AS total_rows
    FROM BillingPolicyInterface
""").show(truncate=False)

print()
print("=== BILLING = 0 COUNT BY EFFECTIVE DATE ===")
stats_flagged.filter(F.col("BillingTotalPremium") == 0)\
    .groupBy("VarianceBucket")\
    .count()\
    .show(truncate=False)

print()
print("=== HOW MANY POLICIES HAVE BILLING = 0 ===")
billing_zero_count = stats_flagged.filter(F.col("BillingTotalPremium") == 0).count()
print(f"Policies with Billing = 0: {billing_zero_count}")
print(f"Remaining mismatches:      {426 - billing_zero_count}")

StatementMeta(, f02b8f7e-465c-4e55-bffb-6218b04fc486, 116, Finished, Available, Finished, False)

=== LATEST BILLING TIMESTAMP ===
+------------------------+----------+
|latest_billing_timestamp|total_rows|
+------------------------+----------+
|2026-06-25 10:30:43.253 |46612     |
+------------------------+----------+


=== BILLING = 0 COUNT BY EFFECTIVE DATE ===
+--------------+-----+
|VarianceBucket|count|
+--------------+-----+
|High          |223  |
|No Variance   |102  |
+--------------+-----+


=== HOW MANY POLICIES HAVE BILLING = 0 ===
Policies with Billing = 0: 325
Remaining mismatches:      101


In [110]:
spark.sql("""
    SELECT DISTINCT
        PolicyNumber,
        CAST(EffectiveDate AS DATE) AS EffectiveDate,
        CAST(ExpirationDate AS DATE) AS ExpirationDate
    FROM policy_versions
    WHERE REGEXP_REPLACE(PolicyNumber, '-[0-9]+$', '') 
          IN ('GAPA006750968', 'GAPA006654200')
    ORDER BY PolicyNumber, EffectiveDate
""").show(truncate=False)

StatementMeta(, f02b8f7e-465c-4e55-bffb-6218b04fc486, 117, Finished, Available, Finished, False)

+---------------+-------------+--------------+
|PolicyNumber   |EffectiveDate|ExpirationDate|
+---------------+-------------+--------------+
|GAPA006654200-1|2025-10-29   |2026-04-29    |
|GAPA006750968  |2025-10-08   |2026-04-08    |
|GAPA006750968-1|2026-04-08   |2026-10-08    |
+---------------+-------------+--------------+

